In [22]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session
from google.cloud import storage

# ==================================================
# 1. CONFIGURATION
# ==================================================

PROJECT_ID = "fintech-lakehouse-dev"
REGION = "us-central1"

CATALOG_NAME = "fintech_lakehouse"
CATALOG_ID = "fintech_lakehouse_bucket"

REST_URI = "https://biglake.googleapis.com/iceberg/v1/restcatalog"

WAREHOUSE = (
    f"bl://projects/{PROJECT_ID}/catalogs/{CATALOG_ID}"
)

# ==================================================
# 2. DATAPROC RUNTIME CONFIGURATION
# ==================================================

session_config = Session()

# Spark 4.x runtime
session_config.runtime_config.version = "3.0"

## JVM DEPENDENCY JARS
BUCKET = "fintech_lakehouse_bucket"
core = [
    "spark-sql-kafka-0-10_2.13-4.0.2.jar",
    "spark-token-provider-kafka-0-10_2.13-4.0.2.jar",
    "kafka-clients-3.9.0.jar",
    "commons-pool2-2.12.0.jar",
    "spark-avro_2.13-4.0.2.jar",
]
auth = [b.name for b in storage.Client().list_blobs(BUCKET, prefix="jars/auth/") if b.name.endswith(".jar")]

jars = [f"gs://{BUCKET}/jars/{j}" for j in core] + [f"gs://{BUCKET}/{n}" for n in auth]
session_config.runtime_config.properties["spark.jars"] = ",".join(jars)
print(len(jars), "jars")

# ==================================================
# 3. ICEBERG CATALOG CONFIGURATION
# ==================================================

builder = (
    DataprocSparkSession.builder
    .appName("FintechLakehouse")
    .dataprocSessionConfig(session_config)

    .config("spark.sql.defaultCatalog", CATALOG_NAME)

    # Iceberg REST catalog
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.type",
        "rest"
    )
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.uri",
        REST_URI
    )
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.warehouse",
        WAREHOUSE
    )

    # GCP project
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.header.x-goog-user-project",
        PROJECT_ID
    )

    # Google authentication
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.rest.auth.type",
        "org.apache.iceberg.gcp.auth.GoogleAuthManager"
    )

    # GCS FileIO
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.io-impl",
        "org.apache.iceberg.gcp.gcs.GCSFileIO"
    )

    # Iceberg SQL extensions
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )

    # Credential vending
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.header.X-Iceberg-Access-Delegation",
        "vended-credentials"
    )
)

# ==================================================
# 4. CREATE SPARK SESSION
# ==================================================

spark = builder.getOrCreate()

print("Spark session initialized!")
print("Spark version:", spark.version)

41 jars
Using existing Dataproc Session (configuration changes may not be applied): https://console.cloud.google.com/dataproc/interactive/us-central1/sc-20261107-145510-vihu3t?project=fintech-lakehouse-dev


Spark session initialized!
Spark version: 4.0.2


In [23]:
spark.sql("""
    CREATE NAMESPACE IF NOT EXISTS
    fintech_lakehouse.bronze
""")

DataFrame[]

In [24]:
spark.sql("""
    SHOW NAMESPACES IN fintech_lakehouse
""").show()

+---------+
|namespace|
+---------+
|   bronze|
|   silver|
+---------+



In [25]:
from pyspark.sql import functions as F
from pyspark.sql.avro.functions import from_avro

PROJECT_ID = "fintech-lakehouse-dev"

TOPIC_NAME = "card-transactions-topic"

CATALOG = 'fintech_lakehouse'

BRONZE_SCHEMA = f"{CATALOG}.bronze"

BRONZE1 = f"{BRONZE_SCHEMA}.raw_transactions"

BUCKET = "fintech_lakehouse_bucket"

CHECKPOINT1 = f"gs://{BUCKET}/checkpoints/bronze1"

print (BRONZE1)

fintech_lakehouse.bronze.raw_transactions


In [26]:
avro_schema = {
    "type": "record",
    "name": "CardTransactionEvent",
    "namespace": "com.fintech.lakehouse.streaming",
    "fields": [
        {"name": "transaction_id", "type": "string"},
        {"name": "customer_id", "type": "string"},
        {"name": "card_type", "type": "string"},
        {"name": "timestamp", "type": "long"},
        {"name": "amount", "type": "double"},
        {"name": "currency", "type": "string", "default": "USD"},
        {"name": "merchant_name", "type": "string"},
        {"name": "merchant_category", "type": "string"},
        {"name": "location_country", "type": "string"},
        {"name": "is_international", "type": "boolean"},
        {"name": "device_fingerprint", "type": "string"},
        {"name": "fraud_risk_score", "type": "float"}
    ]
}

import json

AVRO_SCHEMA_JSON = json.dumps(avro_schema)

BRONZE1

In [27]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE1} (
    kafka_key STRING,
    payload BINARY,
    topic STRING,
    partition INT,
    offset BIGINT,
    kafka_timestamp TIMESTAMP,
    ingestion_timestamp TIMESTAMP
)
USING ICEBERG
""")

DataFrame[]

In [28]:
BOOTSTRAP_SERVERS = (
    "bootstrap-m2fh0vmmtugrztbwci63yeoe6."
    "8d4c02221.us-central1.managedkafka.s.cloud.goog:9092"
)

kafka_options = {
    "kafka.bootstrap.servers": BOOTSTRAP_SERVERS,
    "subscribe": TOPIC_NAME,
    "startingOffsets": "latest",
    "maxOffsetsPerTrigger": "5000",

    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "OAUTHBEARER",

    "kafka.sasl.login.callback.handler.class":
        "com.google.cloud.hosted.kafka.auth.GcpLoginCallbackHandler",

    "kafka.sasl.jaas.config":
        "org.apache.kafka.common.security.oauthbearer.OAuthBearerLoginModule required;"
}

kafka_stream = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

bronze1_stream = kafka_stream.select(
    F.col("key").cast("string").alias("kafka_key"),
    F.col("value").alias("payload"),
    F.col("topic"),
    F.col("partition"),
    F.col("offset"),
    F.col("timestamp").alias("kafka_timestamp"),
    F.current_timestamp().alias("ingestion_timestamp")
)

In [29]:
query_bronze1 = (
    bronze1_stream.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT1)
    .trigger(processingTime="1 minute")
    .toTable(BRONZE1)
)

In [30]:
print("Bronze 1:", query_bronze1.isActive)

Bronze 1: True


In [31]:
spark.table(BRONZE1).show(10, truncate=False)

+---------+-------+-----+---------+------+---------------+-------------------+
|kafka_key|payload|topic|partition|offset|kafka_timestamp|ingestion_timestamp|
+---------+-------+-----+---------+------+---------------+-------------------+
+---------+-------+-----+---------+------+---------------+-------------------+



In [32]:
for name, q in [("bronze1", query_bronze1)]:
    print(name, q.status)
    print(name, "exception:", q.exception())

bronze1 {'message': 'Getting offsets from KafkaV2[Subscribe[card-transactions-topic]]', 'isDataAvailable': False, 'isTriggerActive': True}


bronze1 exception: None


In [34]:
# query_bronze1.stop()